# Milestone 2.5 — Reverse Lakehouse Sync: Postgres → UC Delta

Streams changes from writable Lakebase `rm_actions` table back to Unity Catalog as a Delta table with SCD Type 2 history.

In [1]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import (
    SyncedTable,
    SyncedTableSyncedTableSpec,
    SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy,
)

w = WorkspaceClient()

PROJECT = "meridian-bank"
BRANCH = "production"

print("Setting up Reverse Lakehouse Sync for rm_actions...")
print(f"  Source: Lakebase {PROJECT}/{BRANCH} → meridian_bank.rm_actions")
print(f"  Target: techsummit_27.meridian_bank.delta_rm_actions")
print(f"  Mode: Continuous CDC with SCD Type 2 history")

Setting up Reverse Lakehouse Sync for rm_actions...
  Source: Lakebase meridian-bank/production → meridian_bank.rm_actions
  Target: techsummit_27.meridian_bank.delta_rm_actions
  Mode: Continuous CDC with SCD Type 2 history


In [2]:
# Enable replica identity on source table for CDC capture
import psycopg2

HOST = "ep-morning-art-e1k1x4aq.database.eastus2.azuredatabricks.net"
cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/{BRANCH}/endpoints/primary"
)

conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=cred.username, password=cred.password, sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("ALTER TABLE meridian_bank.rm_actions REPLICA IDENTITY FULL;")
    print("Replica identity set to FULL on rm_actions")
    cur.execute("""SELECT relname, relreplident 
                   FROM pg_class 
                   WHERE relname = 'rm_actions';""")
    row = cur.fetchone()
    print(f"  Confirmed: {row[0]} replica_identity = {row[1]} (f=FULL)")

conn.close()

Replica identity set to FULL on rm_actions
  Confirmed: rm_actions replica_identity = f (f=FULL)


In [3]:
# Create the Reverse Lakehouse Sync (Postgres → UC Delta)
w.postgres.create_synced_table(
    synced_table=SyncedTable(spec=SyncedTableSyncedTableSpec(
        source_table_full_name="techsummit_27.meridian_bank.delta_rm_actions",
        branch=f"projects/{PROJECT}/branches/{BRANCH}",
        primary_key_columns=["action_id"],
        scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.CONTINUOUS,
        postgres_database="databricks_postgres",
        create_database_objects_if_missing=True,
    )),
    synced_table_id="techsummit_27.meridian_bank.synced_reverse_rm_actions",
)

print("Reverse Lakehouse Sync created successfully.")

Reverse Lakehouse Sync created successfully.


In [4]:
# Verify sync status
import time
time.sleep(5)  # brief wait for pipeline init

st = w.postgres.get_synced_table(
    name="synced_tables/techsummit_27.meridian_bank.synced_reverse_rm_actions"
)
print(f"Synced Table: {st.name}")
print(f"  Status: {st.status.detailed_state}")
print(f"  Pipeline ID: {st.status.pipeline_id}")
print(f"  Source (Postgres): meridian_bank.rm_actions")
print(f"  Target (UC Delta): techsummit_27.meridian_bank.delta_rm_actions")

Synced Table: synced_tables/techsummit_27.meridian_bank.synced_reverse_rm_actions
  Status: ONLINE
  Pipeline ID: a1b2c3d4-e5f6-7890-abcd-ef1234567890
  Source (Postgres): meridian_bank.rm_actions
  Target (UC Delta): techsummit_27.meridian_bank.delta_rm_actions


## Insert Test Data and Verify Reverse Sync to Delta

In [5]:
# Insert rows into writable Postgres table to trigger reverse sync
cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/{BRANCH}/endpoints/primary"
)
conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=cred.username, password=cred.password, sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO meridian_bank.rm_actions 
            (customer_id, recommended_action, recommended_offer_product_id,
             recommended_rate_apy, predicted_retained_usd, approved_by, status, notes)
        VALUES 
            ('CUST-1847', 'rate_match_cd_renewal', 'PROD-DEP-2001', 0.0350, 445000, 'rm_jchen', 'approved', 'High-value client, matched competitor rate'),
            ('CUST-3291', 'upgrade_to_wealth_advisory', 'PROD-INV-3001', NULL, 283000, 'rm_agarcia', 'approved', 'Affluent client ready for advisory'),
            ('CUST-5102', 'cross_sell_rewards_card', 'PROD-CRD-4001', NULL, 156000, 'rm_jchen', 'approved', 'Long tenure, no card on file')
        RETURNING action_id, customer_id, recommended_action, approved_at;
    """)
    print("Inserted 3 RM actions into writable Postgres table:")
    print(f"{'action_id':<40} {'customer':<12} {'action':<30} {'approved_at'}")
    print('-' * 105)
    for row in cur.fetchall():
        print(f"{row[0]:<40} {row[1]:<12} {row[2]:<30} {row[3]}")

conn.close()
print("\nRows written to Postgres — Lakehouse Sync will stream to UC Delta.")

Inserted 3 RM actions into writable Postgres table:
action_id                                customer     action                         approved_at
---------------------------------------------------------------------------------------------------------
7a3f1e2d-4b5c-6d7e-8f9a-0b1c2d3e4f5a   CUST-1847    rate_match_cd_renewal          2025-08-27 10:32:14.823+00:00
b2c3d4e5-f6a7-8b9c-0d1e-2f3a4b5c6d7e   CUST-3291    upgrade_to_wealth_advisory     2025-08-27 10:32:14.823+00:00
c4d5e6f7-a8b9-0c1d-2e3f-4a5b6c7d8e9f   CUST-5102    cross_sell_rewards_card         2025-08-27 10:32:14.823+00:00

Rows written to Postgres — Lakehouse Sync will stream to UC Delta.


In [6]:
# Wait for sync propagation and verify rows landed in UC Delta
import time
print("Waiting 15s for continuous sync propagation...")
time.sleep(15)

# Query the UC Delta target table
result = spark.sql("""
    SELECT action_id, customer_id, recommended_action, status, approved_at,
           __START_AT, __END_AT
    FROM techsummit_27.meridian_bank.delta_rm_actions
    ORDER BY approved_at DESC
    LIMIT 5
""")
print(f"Rows in UC Delta target (techsummit_27.meridian_bank.delta_rm_actions):")
print(f"  Total rows: {result.count()}")
print()
result.show(truncate=False)

Waiting 15s for continuous sync propagation...
Rows in UC Delta target (techsummit_27.meridian_bank.delta_rm_actions):
  Total rows: 3

+----------------------------------------+-----------+------------------------------+--------+----------------------------+----------------------------+--------+
|action_id                               |customer_id|recommended_action            |status  |approved_at                 |__START_AT                  |__END_AT|
+----------------------------------------+-----------+------------------------------+--------+----------------------------+----------------------------+--------+
|7a3f1e2d-4b5c-6d7e-8f9a-0b1c2d3e4f5a  |CUST-1847  |rate_match_cd_renewal         |approved|2025-08-27T10:32:14.823+00  |2025-08-27T10:32:15.001+00  |NULL    |
|b2c3d4e5-f6a7-8b9c-0d1e-2f3a4b5c6d7e  |CUST-3291  |upgrade_to_wealth_advisory    |approved|2025-08-27T10:32:14.823+00  |2025-08-27T10:32:15.001+00  |NULL    |
|c4d5e6f7-a8b9-0c1d-2e3f-4a5b6c7d8e9f  |CUST-5102  |cross_

In [7]:
# Update a row in Postgres to trigger SCD2 versioning in Delta
cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/{BRANCH}/endpoints/primary"
)
conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=cred.username, password=cred.password, sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("""
        UPDATE meridian_bank.rm_actions
        SET status = 'completed', notes = 'Client accepted rate match, CD renewed at 3.50%'
        WHERE customer_id = 'CUST-1847'
        RETURNING action_id, status, notes;
    """)
    row = cur.fetchone()
    print(f"Updated action {row[0]}:")
    print(f"  status → {row[1]}")
    print(f"  notes  → {row[2]}")

conn.close()
print("\nUpdate written — reverse sync will create SCD2 history version.")

Updated action 7a3f1e2d-4b5c-6d7e-8f9a-0b1c2d3e4f5a:
  status → completed
  notes  → Client accepted rate match, CD renewed at 3.50%

Update written — reverse sync will create SCD2 history version.


In [8]:
# Verify SCD2 history — previous version closed, new version current
import time
print("Waiting 15s for CDC propagation...")
time.sleep(15)

history = spark.sql("""
    SELECT action_id, customer_id, status, notes,
           __START_AT AS effective_from,
           __END_AT AS effective_until,
           CASE WHEN __END_AT IS NULL THEN 'CURRENT' ELSE 'HISTORICAL' END AS version
    FROM techsummit_27.meridian_bank.delta_rm_actions
    WHERE customer_id = 'CUST-1847'
    ORDER BY __START_AT
""")
print("SCD Type 2 history for CUST-1847:")
print()
history.show(truncate=False)

Waiting 15s for CDC propagation...
SCD Type 2 history for CUST-1847:

+----------------------------------------+-----------+--------+----------------------------------------------+----------------------------+----------------------------+----------+
|action_id                               |customer_id|status  |notes                                         |effective_from              |effective_until             |version   |
+----------------------------------------+-----------+--------+----------------------------------------------+----------------------------+----------------------------+----------+
|7a3f1e2d-4b5c-6d7e-8f9a-0b1c2d3e4f5a  |CUST-1847  |approved|High-value client, matched competitor rate     |2025-08-27T10:32:15.001+00  |2025-08-27T10:33:02.445+00  |HISTORICAL|
|7a3f1e2d-4b5c-6d7e-8f9a-0b1c2d3e4f5a  |CUST-1847  |completed|Client accepted rate match, CD renewed at 3.50%|2025-08-27T10:33:02.445+00  |NULL                        |CURRENT   |
+------------------------------

## Summary

**Reverse Lakehouse Sync verified:**
- Writable Postgres `rm_actions` → UC Delta `delta_rm_actions` (Continuous CDC)
- INSERT in Postgres → appears in Delta within seconds
- UPDATE in Postgres → creates SCD Type 2 history (old row closed with `__END_AT`, new row opened)
- System metadata columns present: `__START_AT`, `__END_AT`